# Results Analysis: LLM vs. Numeric Performance

This notebook provides a comprehensive comparative analysis of video popularity predictions across various LLM-based approaches and statistical benchmarks. We evaluate accuracy, descriptive statistics, bias, and the impact of qualitative understanding on quantitative performance.

## Methodology

We compare the following models:
1. **Global LLM**: Zero-shot predictions using high-level channel success drivers.
2. **Dimension LLM**: Zero-shot predictions using PCA dimension values and their qualitative definitions.
3. **Feature Evaluation LLM**: Predictions where the LLM first labels specific dimension contributions.
4. **Guided Inference LLM**: Predictions where the LLM is provided with qualitative 'hints' mapped from PCA embeddings.
5. **Numeric (OLS)**: The baseline statistical model trained on PCA embeddings.
6. **Baselines**: Mean and Median views from the training dataset.

**Normalization**: Since channels have vastly different view scales, we normalize video-level predictions by subtracting the channel's central tendency (mean or median) calculated from training data. This allows for dataset-wide aggregation and evaluation.

## 1. Setup and Configuration

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_absolute_error, r2_score
from google.colab import drive

drive.mount('/content/drive')

# --- CONFIGURATION ---
NORM_METHOD = 'median' # Options: 'median' or 'mean'
# ---------------------

BASE_PATH = '/content/drive/MyDrive/numeric_inference_outputs/'
FILES = {
    'eval_data': 'top_significant_channels_eval.json',
    'global': 'llm_global_results.json',
    'dim': 'llm_dimension_results.json',
    'feature': 'feature_evaluation_results.json',
    'guided': 'guided_inference_results.json'
}

def load_json(filename):
    with open(os.path.join(BASE_PATH, filename), 'r') as f:
        return json.load(f)

eval_dataset = load_json(FILES['eval_data'])
global_res = load_json(FILES['global'])
dim_res = load_json(FILES['dim'])
feature_res = load_json(FILES['feature'])
guided_res = load_json(FILES['guided'])

## 2. Data Alignment and Normalization

We flatten the results into a single video-level DataFrame, ensuring all models are compared on the same set of testing videos.

In [ ]:
records = []

for channel in eval_dataset:
    cid = channel['channel_id']
    cname = channel['channel_name']
    
    # Calculate channel-level baselines from training data
    train_log_views = np.log1p([v['actual_views'] for v in channel['train_videos']])
    ch_mean = np.mean(train_log_views)
    ch_median = np.median(train_log_views)
    ch_std = np.std(train_log_views)
    ch_baseline = ch_median if NORM_METHOD == 'median' else ch_mean

    # Map guided results for quick lookup
    guided_map = {r['video_id']: r['predicted_log_views'] for r in guided_res if r['channel_id'] == cid}
    
    # Map feature results for quick lookup
    f_res_list = next((c['results'] for c in feature_res if c['channel_id'] == cid), [])
    feature_map = {r['video_id']: r for r in f_res_list}

    for v in channel['test_videos']:
        vid = v['video_id']
        title_lower = v['title'].lower()
        actual = np.log1p(v['actual_views'])
        numeric = np.log1p(v['predicted_views'])
        
        # Retrieve predictions, handle missing batches
        g_pred = global_res.get(title_lower, {}).get('predicted_log_views')
        d_pred = dim_res.get(title_lower, {}).get('predicted_log_views')
        f_obj = feature_map.get(vid, {})
        f_pred = f_obj.get('predicted_log_views')
        gu_pred = guided_map.get(vid)
        
        records.append({
            'video_id': vid,
            'channel_name': cname,
            'actual': actual,
            'norm_actual': actual - ch_baseline,
            'numeric': numeric,
            'norm_numeric': numeric - ch_baseline,
            'global': g_pred,
            'norm_global': (g_pred - ch_baseline) if g_pred is not None else None,
            'dimension': d_pred,
            'norm_dimension': (d_pred - ch_baseline) if d_pred is not None else None,
            'feature': f_pred,
            'norm_feature': (f_pred - ch_baseline) if f_pred is not None else None,
            'guided': gu_pred,
            'norm_guided': (gu_pred - ch_baseline) if gu_pred is not None else None,
            'baseline_mean': ch_mean,
            'baseline_median': ch_median,
            'label_error': f_obj.get('mean_label_error'),
            'train_std': ch_std
        })

df = pd.DataFrame(records)
print(f"Created dataset with {len(df)} video records across {df['channel_name'].nunique()} channels.")

### Distribution of Normalized Videos

Visualizing the target variable distribution after normalization.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['norm_actual'], kde=True, color='purple')
plt.title(f"Distribution of Actual Log-Views (Normalized by {NORM_METHOD.capitalize()})")
plt.xlabel("Normalized Log-Views")
plt.axvline(0, color='red', linestyle='--', label='Channel Baseline')
plt.legend()
plt.show()

## 3. Accuracy Benchmark

This section compares the Mean Absolute Error (MAE) of all models. We evaluate if the qualitative approaches can compete with the numeric OLS models and baselines.

In [ ]:
def calc_mae(group, pred_col):
    valid = group[group[pred_col].notnull()]
    if len(valid) == 0: return None
    return mean_absolute_error(valid['actual'], valid[pred_col])

models = ['global', 'dimension', 'feature', 'guided', 'numeric', 'baseline_mean', 'baseline_median']
channel_mae = df.groupby('channel_name').apply(lambda x: pd.Series({
    m: calc_mae(x, m) for m in models
})).reset_index()

print("MAE by Channel:")
display(channel_mae)

# Dataset-level metrics using normalized values
global_metrics = []
for m in models:
    norm_col = f"norm_{m}" if f"norm_{m}" in df.columns else m
    valid = df[df[norm_col].notnull()]
    mae = mean_absolute_error(valid['norm_actual'], valid[norm_col])
    r2 = r2_score(valid['norm_actual'], valid[norm_col])
    
    # P-value for predictive value (Correlation with actual)
    corr, pval = stats.pearsonr(valid['norm_actual'], valid[norm_col])
    
    global_metrics.append({
        'Model': m,
        'MAE (Normalized)': mae,
        'R2': r2,
        'p-value': pval
    })

df_global_metrics = pd.DataFrame(global_metrics)
print("\nDataset-Level Normalized Metrics:")
display(df_global_metrics)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=df_global_metrics.sort_values('MAE (Normalized)'), x='Model', y='MAE (Normalized)', palette='viridis')
plt.title("Model Comparison: Normalized MAE (Lower is better)")
plt.xticks(rotation=45)
plt.show()

## 4. Descriptive Analysis: Bias and Dispersion

We examine the statistical properties of the predictions. 
- **Bias**: Measured as the mean error (Predicted - Actual). A positive bias indicates over-prediction.
- **Dispersion**: Measured as the variance of predictions. High variance indicates a model that predicts wide swings in popularity, whereas low variance indicates 'safe' or compact predictions near the mean.

In [ ]:
desc_stats = []

for m in ['global', 'dimension', 'feature', 'guided', 'numeric']:
    valid = df[df[m].notnull()]
    bias = (valid[m] - valid['actual']).mean()
    variance = valid[m].var()
    desc_stats.append({
        'Model': m,
        'Mean Prediction': valid[m].mean(),
        'Bias': bias,
        'Variance': variance,
        'Std Dev': np.sqrt(variance)
    })

# Add ground truth stats
desc_stats.append({
    'Model': 'ACTUAL (Test Set)',
    'Mean Prediction': df['actual'].mean(),
    'Bias': 0,
    'Variance': df['actual'].var(),
    'Std Dev': df['actual'].std()
})

df_desc = pd.DataFrame(desc_stats)
print("Descriptive Statistics Comparison:")
display(df_desc)

# Plotting Distributions
plt.figure(figsize=(14, 8))
plot_models = ['norm_actual', 'norm_numeric', 'norm_global', 'norm_guided']
for col in plot_models:
    sns.kdeplot(df[col].dropna(), label=col.replace('norm_', '').upper(), fill=True, alpha=0.1)

plt.title("Comparison of Prediction Distributions (Normalized)")
plt.xlabel("Deviation from Channel Baseline")
plt.axvline(0, color='black', linestyle='--')
plt.legend()
plt.show()

## 5. Correlation Analysis

We analyze how well the LLM predictions correlate with the Numeric (OLS) model. High correlation suggests that the LLM is capturing similar semantic signals as the statistical model.

In [ ]:
corr_metrics = []
llm_models = ['global', 'dimension', 'feature', 'guided']

for m in llm_models:
    valid = df[df[m].notnull()]
    corr, p = stats.pearsonr(valid[m], valid['numeric'])
    corr_metrics.append({'Model': m, 'Correlation with OLS': corr, 'p-value': p})

df_corr = pd.DataFrame(corr_metrics)
print("Global Correlation with Numeric Model:")
display(df_corr)

# Channel level correlation table
chan_corr = df.groupby('channel_name').apply(lambda x: pd.Series({
    m: stats.pearsonr(x[x[m].notnull()][m], x[x[m].notnull()]['numeric'])[0] if len(x[x[m].notnull()]) > 1 else None
    for m in llm_models
}))
print("\nCorrelation with OLS by Channel:")
display(chan_corr)

# Visualize Correlation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.regplot(data=df, x='numeric', y='guided', ax=axes[0], scatter_kws={'alpha':0.2}, line_kws={'color':'red'})
axes[0].set_title("Numeric OLS vs Guided LLM Predictions")

sns.heatmap(chan_corr.astype(float), annot=True, cmap='coolwarm', ax=axes[1])
axes[1].set_title("Channel-Level Correlation Heatmap")
plt.show()

## 6. Label Accuracy Impact

Utilizing data from the `feature-evaluation` notebook, we test the hypothesis that better qualitative understanding (lower label error) leads to better quantitative prediction accuracy.

In [ ]:
# Drop videos without label error data (feature-evaluation split)
df_feat = df[df['label_error'].notnull()].copy()
df_feat['pred_error'] = (df_feat['feature'] - df_feat['actual']).abs()

print(f"Analyzing {len(df_feat)} videos from feature-evaluation.")

# Segment by Accuracy
median_err = df_feat['label_error'].median()
df_feat['label_quality'] = df_feat['label_error'].apply(lambda x: 'Accurate Labels' if x <= median_err else 'Inaccurate Labels')

quality_comparison = df_feat.groupby('label_quality').agg({
    'pred_error': ['mean', 'std'],
    'feature': 'var'
})
display(quality_comparison)

# Correlation between Label Error and Prediction Error
corr, p = stats.pearsonr(df_feat['label_error'], df_feat['pred_error'])
print(f"\nCorrelation between Label Error and Prediction Error: {corr:.4f} (p={p:.2e})")

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_feat, x='label_quality', y='pred_error', palette='Set2')
plt.title("Prediction Error: Accurate vs Inaccurate Dimension Labeling")
plt.ylabel("Absolute Prediction Error (Log-Views)")
plt.show()

## 7. Conclusions and Interpretation

### Summary of Findings
1. **Model Performance**: [Insert observation about which model had the lowest MAE]
2. **Bias and Variance**: [Insert observation about whether LLMs over/under predict and how dispersed they are compared to OLS]
3. **Qualitative Link**: [Insert observation about the correlation between dimension labeling and prediction accuracy]

### Final Thoughts
The results demonstrate that...